# 01 — Inventário e qualidade dos dados (Olist)

**Objetivo:** antes de qualquer cálculo de KPI, conhecer os arquivos, colunas, tipos, chaves, valores ausentes, duplicatas e inconsistências, e validar a cardinalidade dos relacionamentos que serão usados nos `merge`s.

Este notebook **não altera** nenhum dado. Ele só lê os CSVs originais e registra as verificações em `data_processed/qualidade_01.pkl`, que será consolidado no `qualidade_dados.csv` final (notebook 07).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import pandas as pd
import numpy as np
import olist_utils as U

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

checks = []  # lista de dicionários -> tabela de qualidade
def registrar(tabela, verificacao, valor, detalhe=""):
    checks.append({"etapa": "01_inventario", "tabela": tabela, "verificacao": verificacao,
                   "valor": valor, "detalhe": detalhe})

## 1. Leitura de todas as tabelas
Datas são convertidas com `errors="coerce"`: valores que não forem datas válidas viram `NaT` e são contados abaixo (não são imputados).

In [2]:
dfs = {name: U.load_raw(name) for name in U.RAW_FILES}
inventario = pd.DataFrame([
    {"tabela": n, "arquivo": U.RAW_FILES[n], "linhas": len(d), "colunas": d.shape[1],
     "memoria_mb": round(d.memory_usage(deep=True).sum() / 1e6, 1)}
    for n, d in dfs.items()
])
for r in inventario.itertuples():
    registrar(r.tabela, "linhas", r.linhas, r.arquivo)
inventario

,tabela,arquivo,linhas,colunas,memoria_mb
0,orders,olist_orders_dataset.csv,99441,8,28.2
1,items,olist_order_items_dataset.csv,112650,7,33.7
2,reviews,olist_order_reviews_dataset.csv,99224,7,31.3
3,customers,olist_customers_dataset.csv,99441,5,36.4
4,sellers,olist_sellers_dataset.csv,3095,4,0.9
5,products,olist_products_dataset.csv,32951,9,7.1
6,payments,olist_order_payments_dataset.csv,103886,5,18.7
7,category_translation,product_category_name_translation.csv,71,2,0.0
8,geolocation,olist_geolocation_dataset.csv,1000163,5,206.3


## 2. Colunas, tipos e valores ausentes

In [3]:
linhas = []
for n, d in dfs.items():
    for c in d.columns:
        nulos = int(d[c].isna().sum())
        linhas.append({"tabela": n, "coluna": c, "tipo": str(d[c].dtype), "nulos": nulos,
                       "pct_nulos": round(nulos / len(d) * 100, 3), "distintos": d[c].nunique()})
        if nulos:
            registrar(n, f"nulos::{c}", nulos, f"{nulos / len(d) * 100:.3f}% das linhas")
colunas = pd.DataFrame(linhas)
colunas[colunas["nulos"] > 0]

,tabela,coluna,tipo,nulos,pct_nulos,distintos
4,orders,order_approved_at,datetime64[ns],160,0.161,90733
5,orders,order_delivered_carrier_date,datetime64[ns],1783,1.793,81018
6,orders,order_delivered_customer_date,datetime64[ns],2965,2.982,95664
18,reviews,review_comment_title,object,87656,88.342,4527
19,reviews,review_comment_message,object,58247,58.703,36159
32,products,product_category_name,object,610,1.851,73
33,products,product_name_lenght,float64,610,1.851,66
34,products,product_description_lenght,float64,610,1.851,2960
35,products,product_photos_qty,float64,610,1.851,19
36,products,product_weight_g,float64,2,0.006,2204


## 3. Chaves candidatas e duplicatas
Verificamos se cada chave esperada é realmente única. Em `reviews`, `review_id` **não** é único e há pedidos com mais de uma avaliação — por isso o notebook 02 escolhe a avaliação mais recente por pedido.

In [4]:
chaves = {
    "orders": ["order_id"], "customers": ["customer_id"], "sellers": ["seller_id"],
    "products": ["product_id"], "items": ["order_id", "order_item_id"],
    "payments": ["order_id", "payment_sequential"], "reviews": ["review_id"],
    "category_translation": ["product_category_name"],
}
res = []
for n, k in chaves.items():
    d = dfs[n]
    dup_chave = int(d.duplicated(k).sum())
    dup_linha = int(d.duplicated().sum())
    res.append({"tabela": n, "chave": "+".join(k), "duplicatas_chave": dup_chave, "linhas_identicas": dup_linha})
    registrar(n, f"duplicatas_chave::{'+'.join(k)}", dup_chave)
    registrar(n, "linhas_totalmente_duplicadas", dup_linha)
rev = dfs["reviews"]
pedidos_multi_review = int((rev.groupby("order_id").size() > 1).sum())
registrar("reviews", "pedidos_com_mais_de_uma_avaliacao", pedidos_multi_review)
registrar("reviews", "review_id_e_order_id_duplicados", int(rev.duplicated(["review_id", "order_id"]).sum()))
registrar("geolocation", "linhas_totalmente_duplicadas", int(dfs["geolocation"].duplicated().sum()),
          "tabela não é usada nos KPIs")
print("Pedidos com mais de uma avaliação:", pedidos_multi_review)
pd.DataFrame(res)

Pedidos com mais de uma avaliação: 547


,tabela,chave,duplicatas_chave,linhas_identicas
0,orders,order_id,0,0
1,customers,customer_id,0,0
2,sellers,seller_id,0,0
3,products,product_id,0,0
4,items,order_id+order_item_id,0,0
5,payments,order_id+payment_sequential,0,0
6,reviews,review_id,814,0
7,category_translation,product_category_name,0,0


## 4. Domínios: status, notas, valores monetários

In [5]:
orders = dfs["orders"]
status = orders["order_status"].value_counts()
for s, v in status.items():
    registrar("orders", f"status::{s}", int(v))
display(status.to_frame("pedidos"))

notas = rev["review_score"].value_counts().sort_index()
fora = int((~rev["review_score"].between(1, 5)).sum())
registrar("reviews", "notas_fora_de_1_5", fora)
display(notas.to_frame("avaliacoes"))

items = dfs["items"]
for c in ["price", "freight_value"]:
    neg = int((items[c] < 0).sum()); zero = int((items[c] == 0).sum())
    registrar("items", f"{c}_negativo", neg)
    registrar("items", f"{c}_zero", zero)
    print(f"{c}: negativos={neg}, zero={zero}, min={items[c].min()}, max={items[c].max()}")
pay = dfs["payments"]
registrar("payments", "payment_value_negativo", int((pay["payment_value"] < 0).sum()))

,pedidos
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


,avaliacoes
review_score,
1,11424
2,3151
3,8179
4,19142
5,57328


price: negativos=0, zero=0, min=0.85, max=6735.0
freight_value: negativos=0, zero=383, min=0.0, max=409.68


## 5. Datas: falhas de conversão e incoerências
Contamos datas que não puderam ser convertidas e sequências impossíveis (ex.: entrega ao cliente antes da compra). Nada é corrigido aqui; o tratamento é decidido no notebook 02 conforme a finalidade de cada KPI.

In [6]:
for n, cols in U.DATE_COLS.items():
    raw = U.load_raw(n, parse_dates=False)
    for c in cols:
        falha = int((dfs[n][c].isna() & raw[c].notna()).sum())
        registrar(n, f"data_invalida::{c}", falha, "texto presente mas não conversível")

o = orders
incoerencias = {
    "aprovacao_antes_da_compra": o["order_approved_at"] < o["order_purchase_timestamp"],
    "postagem_antes_da_compra": o["order_delivered_carrier_date"] < o["order_purchase_timestamp"],
    "entrega_cliente_antes_da_postagem": o["order_delivered_customer_date"] < o["order_delivered_carrier_date"],
    "entrega_cliente_antes_da_compra": o["order_delivered_customer_date"] < o["order_purchase_timestamp"],
    "delivered_sem_data_entrega_cliente": (o["order_status"] == "delivered") & o["order_delivered_customer_date"].isna(),
    "delivered_sem_data_postagem": (o["order_status"] == "delivered") & o["order_delivered_carrier_date"].isna(),
}
inc = pd.Series({k: int(v.sum()) for k, v in incoerencias.items()})
for k, v in inc.items():
    registrar("orders", k, v)
print("Intervalo de compras:", o["order_purchase_timestamp"].min(), "->", o["order_purchase_timestamp"].max())
inc.to_frame("pedidos")

Intervalo de compras: 2016-09-04 21:15:19 -> 2018-10-17 17:30:18


,pedidos
aprovacao_antes_da_compra,0
postagem_antes_da_compra,166
entrega_cliente_antes_da_postagem,23
entrega_cliente_antes_da_compra,0
delivered_sem_data_entrega_cliente,8
delivered_sem_data_postagem,2


In [7]:
# Pedidos por ano/mês de compra (mostra que 2018 vai só até agosto/setembro)
o.assign(ano=o["order_purchase_timestamp"].dt.year, mes=o["order_purchase_timestamp"].dt.month) \
 .pivot_table(index="mes", columns="ano", values="order_id", aggfunc="count")

ano,2016,2017,2018
mes,,,
1,NaN,800.0,7269.0
2,NaN,1780.0,6728.0
3,NaN,2682.0,7211.0
4,NaN,2404.0,6939.0
5,NaN,3700.0,6873.0
6,NaN,3245.0,6167.0
7,NaN,4026.0,6292.0
8,NaN,4331.0,6512.0
9,4.0,4285.0,16.0


## 6. Integridade referencial e cardinalidade dos joins
Antes de unir tabelas, conferimos: (a) se todas as chaves estrangeiras existem na tabela de destino; (b) a relação (1:1, 1:N) para saber onde um `merge` poderia multiplicar linhas — principalmente o frete.

In [8]:
def cobertura(origem, col_o, destino, col_d):
    faltando = ~dfs[origem][col_o].isin(dfs[destino][col_d])
    return int(faltando.sum())

ri = {
    "orders.customer_id -> customers": cobertura("orders", "customer_id", "customers", "customer_id"),
    "items.order_id -> orders": cobertura("items", "order_id", "orders", "order_id"),
    "items.seller_id -> sellers": cobertura("items", "seller_id", "sellers", "seller_id"),
    "items.product_id -> products": cobertura("items", "product_id", "products", "product_id"),
    "reviews.order_id -> orders": cobertura("reviews", "order_id", "orders", "order_id"),
    "products.category -> translation": int((dfs["products"]["product_category_name"].notna()
                                             & ~dfs["products"]["product_category_name"].isin(
                                                 dfs["category_translation"]["product_category_name"])).sum()),
}
for k, v in ri.items():
    registrar("relacionamentos", f"chave_sem_correspondencia::{k}", v)

card = {
    "itens por pedido (max)": int(items.groupby("order_id").size().max()),
    "sellers distintos por pedido (max)": int(items.groupby("order_id")["seller_id"].nunique().max()),
    "pedidos com >1 seller": int((items.groupby("order_id")["seller_id"].nunique() > 1).sum()),
    "avaliacoes por pedido (max)": int(rev.groupby("order_id").size().max()),
    "pagamentos por pedido (max)": int(pay.groupby("order_id").size().max()),
    "pedidos sem itens": int((~orders["order_id"].isin(items["order_id"])).sum()),
    "pedidos sem avaliação": int((~orders["order_id"].isin(rev["order_id"])).sum()),
    "orders.customer_id duplicado": int(orders["customer_id"].duplicated().sum()),
}
for k, v in card.items():
    registrar("relacionamentos", k, v)
pd.concat([pd.Series(ri), pd.Series(card)]).to_frame("valor")

,valor
orders.customer_id -> customers,0
items.order_id -> orders,0
items.seller_id -> sellers,0
items.product_id -> products,0
reviews.order_id -> orders,0
products.category -> translation,13
itens por pedido (max),21
sellers distintos por pedido (max),5
pedidos com >1 seller,1278
avaliacoes por pedido (max),3


## 7. UFs presentes vs. mapa de regiões

In [9]:
ufs = set(dfs["customers"]["customer_state"].dropna().unique())
desconhecidas = sorted(ufs - set(U.UF_REGIAO))
registrar("customers", "ufs_distintas", len(ufs))
registrar("customers", "ufs_fora_do_mapa_regiao", len(desconhecidas), ",".join(desconhecidas))
print("UFs de clientes:", len(ufs), "| fora do mapa:", desconhecidas)
assert not desconhecidas

UFs de clientes: 27 | fora do mapa: []


## 8. Salvando o registro de qualidade

In [10]:
qualidade = pd.DataFrame(checks)
qualidade["valor"] = pd.to_numeric(qualidade["valor"])
U.save_processed(qualidade, "qualidade_01")
U.save_processed(colunas, "inventario_colunas")
print(len(qualidade), "verificações registradas")
qualidade.head(20)

85 verificações registradas


,etapa,tabela,verificacao,valor,detalhe
0,01_inventario,orders,linhas,99441,olist_orders_dataset.csv
1,01_inventario,items,linhas,112650,olist_order_items_dataset.csv
2,01_inventario,reviews,linhas,99224,olist_order_reviews_dataset.csv
3,01_inventario,customers,linhas,99441,olist_customers_dataset.csv
4,01_inventario,sellers,linhas,3095,olist_sellers_dataset.csv
5,01_inventario,products,linhas,32951,olist_products_dataset.csv
6,01_inventario,payments,linhas,103886,olist_order_payments_dataset.csv
7,01_inventario,category_translation,linhas,71,product_category_name_translation.csv
8,01_inventario,geolocation,linhas,1000163,olist_geolocation_dataset.csv
9,01_inventario,orders,nulos::order_approved_at,160,0.161% das linhas


## Conclusões deste notebook
- Os 9 arquivos esperados existem e têm o schema padrão do dataset Olist.
- `review_id` não é chave única e existem pedidos com mais de uma avaliação → regra da avaliação mais recente (notebook 02).
- Um pedido pode ter vários itens e vários sellers → o frete deve ser somado **no nível do item**, e o KPI de postagem por pedido precisa de agregação (menor `shipping_limit_date`).
- Há pedidos `delivered` sem data de postagem ou de entrega ao cliente; eles serão excluídos **apenas** do denominador do KPI que depende daquela data.
- A tabela `geolocation` não é necessária para os KPIs (a região vem da UF do cliente).